# 使用 Flask 实现低延时图传
本章节介绍如何使用 Flask 建立一个 Web 应用，用于显示机器人摄像头的实时画面，由于 Web 应用具有可跨平台的特性，用户可以在手机/PC/平板等设备上通过浏览器来观看摄像头的实时画面，实现无线图传功能。

## 什么是 Flask？ 
Flask 是一个轻量级的Web应用框架，用于使用 Python 快速构建Web应用。
- 轻量级：Flask 是一个轻量级框架，它的核心库相对较小，但具有足够的灵活性和可扩展性，使得开发者可以选择添加需要的扩展和库。
- 简单易用：Flask 设计简单，容易上手。它的API清晰明了，文档详尽，使得开发者能够迅速开始并快速构建Web应用。
- 路由系统：Flask使用装饰器来定义URL路由，将请求映射到相应的处理函数。这使得创建不同页面和处理不同请求变得直观而简单。
- 模板引擎：Flask集成了 Jinja2 模板引擎，使得在应用中构建动态内容变得更加容易。模板引擎允许你在HTML中嵌入动态生成的内容。
- 集成开发服务器：Flask带有一个简单的集成开发服务器，方便开发和调试。然而，在生产环境中，建议使用更强大的Web服务器，如 Gunicorn 或 uWSGI。
- 插件和扩展：Flask支持许多插件和扩展，以便添加额外的功能，如数据库集成、身份验证、表单处理等。
- RESTful支持：Flask对RESTful风格的API提供了良好的支持，使得构建和设计RESTful API变得简单。
- WSGI兼容：Flask 是基于WSGI（Web Server Gateway Interface）的，这使得它能够在许多符合WSGI标准的Web服务器上运行。
- 社区活跃：Flask拥有庞大且活跃的社区，这意味着你可以轻松地找到大量的文档、教程和第三方扩展，以及得到支持。

## 准备工作
由于产品开机默认会自动运行主程序，主程序会占用摄像头资源，这种情况下是不能使用本教程的，需要结束主程序或禁止主程序自动运行后再重新启动机器人。

这里需要注意的是，由于机器人主程序中使用了多线程且由 service 配置开机自动运行，所以常规的 sudo killall python 的方法通常是不起作用的，所以我们这里介绍禁用主程序自动运行的方法。

如果你已经禁用了机器人主程序的开机自动运行，则不需要执行下面的`结束主程序`章节。

### 结束主程序
1. 点击上方本页面选项卡旁边的 “+”号，会打开一个新的名为 Launcher 的选项卡。
2. 点击 Other 内的 Terminal，打开终端窗口。
3. 在终端窗口内输入 `bash` 后按回车。
4. 现在你可以使用 Bash Shell 来控制机器人了。
5. 输入命令： `systemctl --user stop ugv-app.service`。
6. 在终端页面，命令执行完后，继续该教程的剩余部分。

## Web 应用例程
### 注意，不能在 Jupyter Lab 中运行下面的代码块
由于 Flask 应用会与 Jupyter Lab 在端口号的使用上产生冲突，所以以下代码不能在 Jupyter Lab 中运行，以下程序存储在 `tutorial_cn` 和 `tutorial_en` 中的名为 `12` 的文件夹内, 在 `12` 文件夹内还有一个名为 `template` 的文件夹用于存储网页资源，以下是例程的运行方法。

1. 用上文介绍的方式来打开终端，此时注意左侧的文件夹路径，新打开的终端默认的路径与左侧的文件路径相同，你需要浏览到 `tutorial_cn` 或 `tutorial_en` 文件夹内，打开终端后输入 `cd 12` 浏览到 `12` 文件夹内。
2. 使用以下命令来启动 Flask Web 应用服务端： `source /home/ws/ugv_rpi/ugv-env/bin/activate && python flask_camera.py`
3. 然后在同一局域网内的设备（也可以是本设备在浏览器中打开一个新的标签页）中打开浏览器，输入树莓派的IP:5000（例如树莓派的IP是192.168.10.104的话，则打开192.168.10.104:5000这个地址），注意`:`需要为英文的冒号。
4. 在终端中使用 Ctrl + C 来结束运行。

### Flask 的程序介绍

In [ ]:
from flask import Flask, render_template, Response, send_from_directory
import threading
import cv_ctrl
import os

# 创建 Flask 应用实例
app = Flask(__name__)

# 创建 OpenCV 功能类实例（用于视频处理）
cvf = cv_ctrl.OpencvFuncs()

@app.route('/')
def index():
    """
    主页路由
    访问 http://<ip>:5000/ 时返回 index.html 页面
    """
    return render_template('index.html')

@app.route('/<path:filename>')
def serve_static(filename):
    """
    静态文件路由
    直接访问文件路径时，从 templates 目录返回对应文件
    """
    return send_from_directory('templates', filename)

@app.route('/settings/<path:filename>')
def serve_static_settings(filename):
    """
    设置页面静态文件路由
    从 templates 目录返回 settings 页面所需的文件
    """
    return send_from_directory('templates', filename)
        
if __name__ == '__main__':
    # 启动摄像头处理线程（守护线程，主程序退出时自动结束）
    cam_thread = threading.Thread(target=cvf.frame_process, daemon=True)
    cam_thread.start()

    # 启动 Flask Web 服务
    # host='0.0.0.0' 表示外网可访问
    # port=5000 表示监听 5000 端口
    # debug=False 关闭调试模式
    app.run(host='0.0.0.0', port=5000, debug=False)

### 以下是代码的一些关键部分的说明

核心是通过 Flask 搭建一个网页服务器，并使用多线程同时运行 OpenCV 视频处理。Flask 负责响应浏览器请求，返回主页面和静态资源;

视频处理逻辑由 `cvf.frame_process()` 在单独的守护线程中持续运行，从而实现前端页面与后台图像处理的并行工作。

### 网页部分介绍

In [ ]:
<!doctype html>
<html lang="en">
<head>
    <!-- Required meta tags -->
    <meta charset="utf-8">
    <title>Live Video Based on Flask</title>
</head>
<body>
    <div class="video">
        <video id="video" autoplay muted playsinline></video>
    </div>
    <div id="message"></div>
    <script type="module" src="./webrtc_reader.js"></script>
    <script>
    const videoElement = document.getElementById('video');
    const message = document.getElementById('message');
    const setMessage = (str) => {
    if (str !== '') {
        videoElement.controls = false;
    } else {
        videoElement.controls = defaultControls;
    }
    message.innerText = str;
    };

    const parseBoolString = (str, defaultVal) => {
    str = (str || '');

    if (['1', 'yes', 'true'].includes(str.toLowerCase())) {
        return true;
    }
    if (['0', 'no', 'false'].includes(str.toLowerCase())) {
        return false;
    }
    return defaultVal;
    };

    const loadAttributesFromQuery = () => {
    const params = new URLSearchParams(window.location.search);
    videoElement.controls = parseBoolString(params.get('controls'), false);
    videoElement.muted = parseBoolString(params.get('muted'), true);
    videoElement.autoplay = parseBoolString(params.get('autoplay'), true);
    videoElement.playsInline = parseBoolString(params.get('playsinline'), false);
    defaultControls = videoElement.controls;
    };

    let stream_url = `http://${window.location.hostname}:8889/cam/`;
    window.addEventListener('load', () => {
    loadAttributesFromQuery();
    new MediaMTXWebRTCReader({
        url: new URL('whep', stream_url) + window.location.search,
        onError: (err) => {
        setMessage(err);
        },
        onTrack: (evt) => {
        setMessage('');
        videoElement.srcObject = evt.streams[0];
        },
    });
    });  

    </script>
</body>
</html>

### 注释

#### **1. HTML 结构**

```html
<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>Live Video Based on Flask</title>
</head>
<body>
    <div class="video">
        <video id="video" autoplay muted playsinline></video>
    </div>
    <div id="message"></div>
    <script type="module" src="./webrtc_reader.js"></script>
    <script> ... </script>
</body>
</html>
```

* `<video>`：用来播放视频流，`autoplay` 自动播放，`muted` 静音（防止浏览器阻止自动播放），`playsinline` 让视频在移动端也能内嵌播放。
* `<div id="message">`：用来显示错误或状态信息。
* `webrtc_reader.js`：外部 JavaScript 模块，里面定义了 `MediaMTXWebRTCReader`，负责处理 WebRTC 数据流。

#### **2. JavaScript 功能**

主要做了三件事：

##### **(1) 处理状态信息**

```javascript
const setMessage = (str) => {
    if (str !== '') {
        videoElement.controls = false;  // 如果有消息，就隐藏视频控制条
    } else {
        videoElement.controls = defaultControls; // 没有消息时恢复控制条状态
    }
    message.innerText = str; // 显示消息
};
```

功能：

* 用于显示错误信息或者清除消息。
* 控制 `<video>` 标签的控件是否显示。

##### **(2) 从 URL 参数读取视频属性**

```javascript
const parseBoolString = (str, defaultVal) => {
    str = (str || '');
    if (['1', 'yes', 'true'].includes(str.toLowerCase())) return true;
    if (['0', 'no', 'false'].includes(str.toLowerCase())) return false;
    return defaultVal;
};

const loadAttributesFromQuery = () => {
    const params = new URLSearchParams(window.location.search);
    videoElement.controls = parseBoolString(params.get('controls'), false);
    videoElement.muted = parseBoolString(params.get('muted'), true);
    videoElement.autoplay = parseBoolString(params.get('autoplay'), true);
    videoElement.playsInline = parseBoolString(params.get('playsinline'), false);
    defaultControls = videoElement.controls;
};
```

功能：

* 根据访问 URL 的查询参数动态设置 `<video>` 的属性，比如：

  ```
  http://localhost:5000?controls=true&muted=false
  ```

  这样会控制视频的播放属性。

##### **(3) 初始化 WebRTC 视频流**

```javascript
let stream_url = `http://${window.location.hostname}:8889/cam/`;
window.addEventListener('load', () => {
    loadAttributesFromQuery();
    new MediaMTXWebRTCReader({
        url: new URL('whep', stream_url) + window.location.search,
        onError: (err) => {
            setMessage(err);
        },
        onTrack: (evt) => {
            setMessage('');
            videoElement.srcObject = evt.streams[0];
        },
    });
});
```

功能：

* 构造视频流的地址，例如 `http://服务器IP:8889/cam/`
* `MediaMTXWebRTCReader`（来自 `webrtc_reader.js`）负责发起 WebRTC 连接：

  * `onError`：如果连接失败，显示错误信息。
  * `onTrack`：收到视频轨道后，把视频流赋值给 `<video>` 标签的 `srcObject`，进行播放。

#### **3. 总结运行流程**

1. 页面加载 → 读取 URL 参数设置视频播放属性。
2. 调用 `MediaMTXWebRTCReader` → 向流媒体服务器（8889 端口）发起 WebRTC 连接。
3. 成功时 → 把流绑定到 `<video>` 播放。
4. 出错时 → 显示错误信息并隐藏视频控件。